In [0]:
# Databricks notebook source
# ============================================================
# 1. CARGA Y LIMPIEZA DE COLUMNAS
# ============================================================
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

DATA_PATH = "/Volumes/workspace/bigdata_daigreco/tp_final_data/bank-additional-full.csv"

# Lectura del CSV
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ";")
    .csv(DATA_PATH)
)

# Limpieza de nombres de columnas (quitar puntos)
clean_columns = [c.replace(".", "_").replace('"', '').strip() for c in df_raw.columns]

df = df_raw
for old_col, new_col in zip(df_raw.columns, clean_columns):
    df = df.withColumnRenamed(old_col, new_col)

# Renombrar y mapear target
df = df.withColumnRenamed("y", "DEPOSIT")
df = df.withColumn(
    "DEPOSIT",
    F.when(F.col("DEPOSIT") == "yes", 1).otherwise(0)
)

print(f"Registros iniciales: {df.count():,}")

# COMMAND ----------

# ============================================================
# 2. DEFINICIÓN DE VARIABLES NUMÉRICAS Y CATEGÓRICAS
# ============================================================
categorical_cols = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome"
]

excluded_cols = ["DEPOSIT"]

numerical_cols = [
    c for c in df.columns
    if c not in categorical_cols + excluded_cols
]

print("Variables categóricas:", len(categorical_cols))
print("Variables numéricas:", len(numerical_cols))

# COMMAND ----------

# ============================================================
# 3. PIPELINE DE PREPROCESAMIENTO (PySpark ML)
# ============================================================
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_index",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_index" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols],
    handleInvalid="keep"
)

assembler_inputs = numerical_cols + [f"{c}_ohe" for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features",
    handleInvalid="keep"
)

preprocessing_pipeline = Pipeline(
    stages=indexers + [encoder, assembler]
)

# COMMAND ----------

# ============================================================
# 4. DIVISION DETERMINÍSTICA TRAIN / TEST (80 / 20)
# ============================================================
# Generamos un ID temporal para garantizar un split reproducible por Hash
df_with_id = df.withColumn("id_temp", F.monotonically_increasing_id())

df_split = df_with_id.withColumn(
    "_split",
    F.pmod(F.xxhash64(F.col("id_temp")), F.lit(100))
)

train_df = df_split.filter(F.col("_split") < 80).drop("_split", "id_temp")
test_df = df_split.filter(F.col("_split") >= 80).drop("_split", "id_temp")

print(f"Train: {train_df.count():,}")
print(f"Test:  {test_df.count():,}")

# COMMAND ----------

# ============================================================
# 5. AJUSTE Y TRANSFORMACIÓN
# ============================================================
preprocessing_model = preprocessing_pipeline.fit(train_df)

train_prepared = preprocessing_model.transform(train_df)
test_prepared = preprocessing_model.transform(test_df)

# COMMAND ----------

# ============================================================
# 6. GUARDAR TABLAS PREPARADAS EN UNITY CATALOG
# ============================================================
TRAIN_TABLE = "workspace.bigdata_daigreco.tp_final_train"
TEST_TABLE = "workspace.bigdata_daigreco.tp_final_test"

(
    train_prepared
    .select("DEPOSIT", "features")
    .write
    .mode("overwrite")
    .saveAsTable(TRAIN_TABLE)
)

(
    test_prepared
    .select("DEPOSIT", "features")
    .write
    .mode("overwrite")
    .saveAsTable(TEST_TABLE)
)

print("✅ Datasets preparados guardados correctamente como Delta Tables:")
print(f"Train: {TRAIN_TABLE}")
print(f"Test:  {TEST_TABLE}")

Registros iniciales: 41,188
Variables categóricas: 10
Variables numéricas: 10
Train: 32,879
Test:  8,309
✅ Datasets preparados guardados correctamente como Delta Tables:
Train: workspace.bigdata_daigreco.tp_final_train
Test:  workspace.bigdata_daigreco.tp_final_test
